# The Pass-Protection Test

A self-contained reproduction of two **separate** exploratory experiments: OL mirror recovery versus rookie pressure rate, and WR/TE movement versus rookie receiving separation. Attach the official NFL Big Data Bowl 2027 dataset. No repository download, Internet access, or raw-data output is required.

The notebook embeds maintained Python modules, writes them to the working directory, and runs training-only preprocessing with temporal holdouts. Tracking is not a validated scouting signal. Outputs and figures contain local research information: review competition sharing rules before publishing generated artifacts.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib
IN_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working/three_second_research") if IN_KAGGLE else Path.cwd() / "notebook_reproduction"
WORK.mkdir(parents=True, exist_ok=True)
print("Reproduction root:", WORK)


In [ ]:
SOURCES = {'src/__init__.py': '"""Combine-to-NFL analysis pipeline."""\n', 'src/load_data.py': '"""Copy official CSVs and convert to Parquet in bounded-memory batches."""\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nROOT = Path(__file__).resolve().parents[1]\nRAW = ROOT / "data" / "raw"\nPROCESSED = ROOT / "data" / "processed"\n\n\ndef convert_csv(source, destination, chunksize=100_000):\n    # Infer numeric fields explicitly; fixed types prevent all-null chunks from\n    # changing the Arrow schema midway through large files.\n    header = pd.read_csv(source, nrows=0).columns\n    string_fields = {\n        "time", "event", "entity_type", "event_id", "drill_type", "drill_name",\n        "display_name", "combine_position", "nfl_position", "birth_date",\n        "college_name", "college_conference", "game_key", "season_type",\n        "home_team_abbr", "visitor_team_abbr", "team_abbr", "lined_up_position",\n        "play_description", "possession_team", "yardline_side", "game_clock",\n        "play_direction", "offense_formation", "receiver_alignment", "pass_result",\n        "play_nullified_by_penalty", "team_coverage_man_zone", "team_coverage_type",\n        "route_ran", "coverage_assignment", "coverage_assignment_at_snap",\n    }\n    bool_fields = {\n        "in_motion_at_ball_snap", "target", "fumble", "fumble_lost", "pressure_allowed",\n        "quick_pressure", "unblocked_pressure", "blitzing", "caused_forced_fumble",\n        "recovered_fumble", "tackle_for_loss",\n    }\n    # Preserve mixed TRUE/FALSE and 0/1 flag encodings as strings.\n    types = {c: "string" if c in string_fields | bool_fields else "Float64" for c in header}\n    rows = 0\n    temporary = destination.with_suffix(".parquet.tmp")\n    try:\n        writer = None\n        try:\n            for chunk in pd.read_csv(source, dtype=types, chunksize=chunksize):\n                table = pa.Table.from_pandas(chunk, preserve_index=False)\n                if writer is None:\n                    writer = pq.ParquetWriter(temporary, table.schema, compression="zstd")\n                writer.write_table(table)\n                rows += len(chunk)\n        finally:\n            if writer is not None:\n                writer.close()\n        temporary.replace(destination)\n    except Exception:\n        temporary.unlink(missing_ok=True)\n        raise\n    return rows\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--source", type=Path, help="Folder containing official CSVs; copies into data/raw")\n    parser.add_argument("--tables", nargs="+", help="CSV stems to convert; default: all")\n    args = parser.parse_args()\n    RAW.mkdir(parents=True, exist_ok=True)\n    PROCESSED.mkdir(parents=True, exist_ok=True)\n    if args.source:\n        for source in args.source.glob("*.csv"):\n            target = RAW / source.name\n            if source.resolve() != target.resolve() and not target.exists():\n                shutil.copy2(source, target)\n    files = sorted(RAW.glob("*.csv"))\n    if args.tables:\n        files = [RAW / f"{name}.csv" for name in args.tables]\n    if not files:\n        raise SystemExit("No CSVs found. Supply --source or populate data/raw.")\n    manifest_path = PROCESSED / "ingestion_manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}\n    for source in files:\n        rows = convert_csv(source, PROCESSED / f"{source.stem}.parquet")\n        manifest[source.name] = {"rows": rows, "source_bytes": source.stat().st_size}\n        print(f"{source.name}: {rows:,} rows", flush=True)\n    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/clean_tracking.py': '"""Clean Combine trajectories without filling gaps or assuming coordinate units."""\nimport json\nimport numpy as np\nimport pandas as pd\nfrom src.load_data import PROCESSED, ROOT\n\nKEYS = [\'draft_year\', \'nfl_id\', \'event_id\', \'drill_name\', \'attempt\']\n\ndef clean_tracking(frame, return_report=False):\n    report={\'source_rows\':len(frame)}\n    frame=frame.loc[frame.entity_type.eq(\'PLAYER\')].copy()\n    report[\'player_rows\']=len(frame)\n    frame[\'time\']=pd.to_datetime(frame.time,errors=\'coerce\')\n    before=len(frame)\n    frame=frame.dropna(subset=KEYS+[\'time\',\'x\',\'y\',\'s\'])\n    finite=np.isfinite(frame[[\'x\',\'y\',\'s\']].to_numpy(dtype=float)).all(axis=1)\n    frame=frame.loc[finite & frame.s.ge(0)]\n    report[\'invalid_or_missing_rows_removed\']=before-len(frame)\n    for key in [\'nfl_id\',\'draft_year\',\'attempt\']:\n        values=frame[key].to_numpy(dtype=float)\n        if not np.isfinite(values).all() or not np.equal(values,np.floor(values)).all():\n            raise ValueError(f\'{key} must contain finite integer-valued identifiers\')\n    duplicate=frame.duplicated(KEYS+[\'time\'],keep=False)\n    if duplicate.any():\n        conflicts=frame.loc[duplicate].groupby(KEYS+[\'time\'],observed=True)[[\'x\',\'y\',\'s\']].nunique().gt(1).any(axis=1)\n        report[\'conflicting_duplicate_timestamps\']=int(conflicts.sum())\n    else:report[\'conflicting_duplicate_timestamps\']=0\n    before=len(frame)\n    # Retain original first-row policy and report conflicts rather than silently average.\n    frame=frame.drop_duplicates(KEYS+[\'time\']).sort_values(KEYS+[\'time\'])\n    report[\'duplicate_rows_removed\']=before-len(frame)\n    frame[\'dt\']=frame.groupby(KEYS,observed=True).time.diff().dt.total_seconds()\n    valid=frame.dt.between(.05,.15)\n    frame[\'signed_acceleration\']=(frame.groupby(KEYS,observed=True).s.diff()/frame.dt).where(valid)\n    report[\'clean_frames\']=len(frame)\n    report[\'invalid_derivative_intervals\']=int((frame.dt.notna()&~valid).sum())\n    frame=frame.reset_index(drop=True)\n    return (frame,report) if return_report else frame\n\ndef main():\n    frame,report=clean_tracking(pd.read_parquet(PROCESSED/\'combine_tracking.parquet\'),return_report=True)\n    frame.to_parquet(PROCESSED/\'combine_tracking_clean.parquet\',index=False)\n    destination=ROOT/\'outputs/research\';destination.mkdir(parents=True,exist_ok=True)\n    (destination/\'tracking_quality.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    print(json.dumps(report,indent=2))\nif __name__==\'__main__\':main()\n', 'src/combine_metrics.py': '"""Summarize attempts, then use medians across repeated attempts per drill."""\nimport pandas as pd\nfrom src.clean_tracking import KEYS\nfrom src.load_data import PROCESSED\n\n\ndef build_metrics(frame):\n    attempts = frame.groupby(KEYS, observed=True).agg(\n        peak_speed=("s", "max"), mean_speed=("s", "mean"),\n        accel_p90=("signed_acceleration", lambda x: x.quantile(0.9)),\n        brake_p10=("signed_acceleration", lambda x: x.quantile(0.1)),\n        frames=("time", "size"), start=("time", "min"), end=("time", "max"),\n    ).reset_index()\n    attempts["duration"] = (attempts.end - attempts.start).dt.total_seconds()\n    attempts = attempts.loc[attempts.frames.ge(10) & attempts.duration.gt(0)]\n    traits = ["peak_speed", "mean_speed", "accel_p90", "brake_p10", "duration"]\n    metrics = attempts.groupby(["draft_year", "nfl_id", "drill_name"], observed=True)[traits].median()\n    wide = metrics.unstack("drill_name")\n    wide.columns = [f"sensor__{drill}__{trait}" for trait, drill in wide.columns]\n    return attempts, wide.reset_index()\n\n\ndef main():\n    attempts, wide = build_metrics(pd.read_parquet(PROCESSED / "combine_tracking_clean.parquet"))\n    attempts.to_parquet(PROCESSED / "combine_attempt_metrics.parquet", index=False)\n    wide.to_parquet(PROCESSED / "combine_metrics.parquet", index=False)\n    print(f"Saved {len(attempts):,} attempts across {len(wide):,} players")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/nfl_outcomes.py': '"""Build rookie regular-season receiver outcomes with explicit opportunity counts."""\nimport pandas as pd\nfrom src.load_data import PROCESSED\n\n\ndef build_outcomes(plays, games, players):\n    games = games.loc[games.season_type.eq("REG"), ["game_id", "season"]]\n    plays = plays.merge(games, on="game_id", validate="many_to_one")\n    plays = plays.merge(players[["nfl_id", "draft_year", "nfl_position"]], on="nfl_id", validate="many_to_one")\n    plays = plays.loc[plays.season.eq(plays.draft_year) & plays.nfl_position.isin(["WR", "TE"])]\n    plays = plays.loc[plays.play_nullified_by_penalty.eq("N")]\n    plays = plays.drop_duplicates(["game_id", "play_id", "nfl_id"])\n    # Route labels distinguish receiving opportunities; separation is measured\n    # at pass-forward and is only summarized when the supplied value exists.\n    routes = plays.loc[plays.route_ran.notna() & plays.route_ran.ne("")].copy()\n    return routes.groupby(["nfl_id", "draft_year"], observed=True).agg(\n        route_plays=("play_id", "size"),\n        separation_observations=("separation_at_pass_forward", "count"),\n        mean_separation=("separation_at_pass_forward", "mean"),\n    ).reset_index()\n\n\ndef main():\n    outcomes = build_outcomes(*(pd.read_parquet(PROCESSED / f"{name}.parquet")\n                                for name in ["player_play", "games", "players"]))\n    outcomes.to_parquet(PROCESSED / "nfl_outcomes.parquet", index=False)\n    print(f"Saved {len(outcomes):,} rookie WR/TE outcomes")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/train_models.py': '"""Compare traditional tests with sensor features using held-out draft cohorts."""\nimport json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import mean_absolute_error, r2_score\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom src.load_data import PROCESSED, ROOT\n\nBASELINE = ["combine_height", "combine_weight", "ten_yd_split", "forty", "vertical",\n            "broad_jump", "three_cone", "short_shuttle"]\n\n\ndef main():\n    tables = {name: pd.read_parquet(PROCESSED / f"{name}.parquet")\n              for name in ["nfl_outcomes", "combine_results", "combine_metrics"]}\n    data = tables["nfl_outcomes"]\n    for name in ["combine_results", "combine_metrics"]:\n        data = data.merge(tables[name], on=["nfl_id", "draft_year"], validate="one_to_one")\n    data = data.loc[data.separation_observations.ge(20) & data.mean_separation.notna()].copy()\n    data["position_TE"] = data.combine_position.eq("TE").astype(int)\n    cohorts = sorted(data.draft_year.unique())\n    if len(cohorts) < 2:\n        raise SystemExit("Need at least two draft cohorts with 20 separation observations per player.")\n    train, test = data.loc[data.draft_year.lt(cohorts[-1])], data.loc[data.draft_year.eq(cohorts[-1])]\n    if len(train) < 10 or len(test) < 2:\n        raise SystemExit("Insufficient players for a meaningful cohort holdout.")\n    baseline = BASELINE + ["position_TE"]\n    # Feature availability is selected on training players only.\n    sensors = [c for c in data if c.startswith("sensor__") and train[c].notna().mean() >= 0.5]\n    if not sensors:\n        raise SystemExit("No sensor features meet training availability threshold.")\n    results, predictions = {}, test[["nfl_id", "draft_year", "mean_separation"]].copy()\n    for name, columns in {"traditional": baseline, "traditional_plus_tracking": baseline + sensors}.items():\n        model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),\n                              StandardScaler(), Ridge(alpha=10.0))\n        model.fit(train[columns], train.mean_separation)\n        pred = model.predict(test[columns])\n        predictions[f"prediction_{name}"] = pred\n        results[name] = {"mae": float(mean_absolute_error(test.mean_separation, pred)),\n                         "r2": float(r2_score(test.mean_separation, pred)), "features": columns}\n    predictions["prediction_train_mean"] = float(train.mean_separation.mean())\n    results["train_mean"] = {"mae": float(mean_absolute_error(test.mean_separation, predictions.prediction_train_mean))}\n    rng = np.random.default_rng(2027)\n    improvement = (np.abs(test.mean_separation.to_numpy() - predictions.prediction_traditional.to_numpy())\n                   - np.abs(test.mean_separation.to_numpy() - predictions.prediction_traditional_plus_tracking.to_numpy()))\n    boot = rng.choice(improvement, size=(2000, len(improvement)), replace=True).mean(axis=1)\n    report = {"train_players": len(train), "test_players": len(test), "held_out_draft_year": int(cohorts[-1]),\n              "models": results, "mae_improvement": float(improvement.mean()),\n              "mae_improvement_bootstrap_95pct": np.quantile(boot, [0.025, 0.975]).tolist(),\n              "limitations": "Exploratory fixed-alpha model; one cohort holdout; observational outcome; no causal claims. Bootstrap describes test-player uncertainty only."}\n    destination = ROOT / "outputs" / "results"\n    destination.mkdir(parents=True, exist_ok=True)\n    (destination / "model_comparison.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    predictions.to_csv(destination / "held_out_predictions.csv", index=False)\n    print(json.dumps({k: v for k, v in report.items() if k != "models"}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'src/validate_results.py': '"""Check basic keys and ranges; extended cohort checks live in scripts.research_checks."""\nimport pandas as pd\nfrom src.load_data import PROCESSED, ROOT\n\n\ndef main():\n    messages = []\n    def require(condition, message):\n        if not condition:\n            raise ValueError(message)\n        messages.append(message)\n    for table in ["players", "combine_results", "combine_metrics", "nfl_outcomes"]:\n        frame = pd.read_parquet(PROCESSED / f"{table}.parquet")\n        keys = ["nfl_id"] if table == "players" else ["nfl_id", "draft_year"]\n        require(not frame.empty, f"{table}: nonempty")\n        require(not frame[keys].isna().any().any(), f"{table}: complete keys")\n        require(not frame.duplicated(keys).any(), f"{table}: unique keys")\n    tracking = pd.read_parquet(PROCESSED / "combine_tracking_clean.parquet")\n    require(tracking.s.ge(0).all(), "tracking: nonnegative speed")\n    require(tracking.time.notna().all(), "tracking: valid timestamps")\n    outcomes = pd.read_parquet(PROCESSED / "nfl_outcomes.parquet")\n    require(outcomes.separation_observations.le(outcomes.route_plays).all(), "outcomes: valid denominators")\n    output = ROOT / "outputs" / "results" / "validation.txt"\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text("\\n".join("PASS " + m for m in messages) + "\\n", encoding="utf-8")\n    print(output.read_text(encoding="utf-8"))\n\n\nif __name__ == "__main__":\n    main()\n', 'src/ol_movement.py': '"""Exploratory dominant-axis reversal and recovery measurements for OL mirror drills."""\nimport json\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import spearmanr\nfrom src.load_data import PROCESSED, ROOT\nfrom src.clean_tracking import KEYS\n\nMIRROR_NAMES = {\'PASS_PRO-MIRROR_DRILL\', \'PASS_PRO_MIRROR_DRILL\'}\n\n\ndef measure_attempt(frame, fraction=0.2):\n    frame = frame.sort_values(\'time\').reset_index(drop=True).copy()\n    times = pd.to_datetime(frame.time)\n    t = (times - times.iloc[0]).dt.total_seconds().to_numpy(dtype=float)\n    xy = frame[[\'x\', \'y\']].to_numpy(dtype=float)\n    if len(xy) < 10 or not np.isfinite(xy).all():\n        raise ValueError(\'An attempt requires at least ten finite coordinate frames.\')\n    centered = xy - np.mean(xy, axis=0)\n    _, singular, vectors = np.linalg.svd(centered, full_matrices=False)\n    axis = vectors[0]\n    if axis[np.argmax(np.abs(axis))] < 0:\n        axis = -axis\n    lateral = centered @ axis\n    dt = np.r_[np.nan, np.diff(t)]\n    valid = np.isfinite(dt) & (dt >= .05) & (dt <= .15)\n    segment = np.cumsum(~valid)\n    velocity = np.full(len(t), np.nan)\n    velocity[valid] = np.diff(lateral, prepend=lateral[0])[valid] / dt[valid]\n    acceleration = np.full(len(t), np.nan)\n    speed = frame.s.to_numpy(dtype=float)\n    acceleration[valid] = np.diff(speed, prepend=speed[0])[valid] / dt[valid]\n    smoothed = pd.Series(velocity).groupby(segment).transform(lambda x: x.rolling(3, center=True, min_periods=2).median()).to_numpy()\n    threshold = .1 * np.nanquantile(np.abs(velocity), .9) if np.isfinite(velocity).any() else np.inf\n    excursion = float(np.quantile(lateral, .9) - np.quantile(lateral, .1))\n    target = fraction * excursion\n    reversals = []\n    if target > 1e-6:\n        for seg in np.unique(segment):\n            indices = np.flatnonzero(segment == seg)\n            signs = np.sign(smoothed[indices])\n            signs[~np.isfinite(smoothed[indices]) | (np.abs(smoothed[indices]) < threshold)] = 0\n            runs = []\n            start = 0\n            for j in range(1, len(indices) + 1):\n                if j == len(indices) or signs[j] != signs[start]:\n                    if signs[start] != 0 and j - start >= 3:\n                        runs.append((int(indices[start]), int(indices[j-1]), int(signs[start])))\n                    start = j\n            for previous, following in zip(runs, runs[1:]):\n                if previous[2] == following[2]:\n                    continue\n                a, b = previous[1], following[0]\n                local = lateral[a:b+1]\n                extremum = int(a + (np.argmax(local) if previous[2] > 0 else np.argmin(local)))\n                if reversals and reversals[-1][\'index\'] == extremum:\n                    continue\n                reversals.append({\'index\': extremum, \'time\': float(t[extremum]), \'direction\': following[2], \'segment\': int(seg)})\n        for j, reversal in enumerate(reversals):\n            idx = reversal[\'index\']\n            same = np.flatnonzero(segment == reversal[\'segment\'])\n            end = int(same[-1])\n            if j+1 < len(reversals) and reversals[j+1][\'segment\'] == reversal[\'segment\']:\n                end = reversals[j+1][\'index\']\n            candidates = np.arange(idx+1, end+1)\n            completed = candidates[(lateral[candidates] - lateral[idx]) * reversal[\'direction\'] >= target]\n            hit = int(completed[0]) if len(completed) else None\n            reversal.update(completed=hit is not None, recovery_index=hit,\n                            delay=float(t[hit]-t[idx]) if hit is not None else None,\n                            observed_until=float(t[end]-t[idx]), target_displacement=target)\n    completed = [r[\'delay\'] for r in reversals if r[\'completed\']]\n    variance = float(singular[0]**2 / np.sum(singular**2)) if np.sum(singular**2) > 0 else 0.\n    summary = {\'reversals\': len(reversals), \'completed\': len(completed),\n               \'recovery_delay\': float(np.median(completed)) if completed else np.nan,\n               \'completion_fraction\': len(completed)/len(reversals) if reversals else np.nan,\n               \'axis_variance_fraction\': variance, \'target_displacement\': target,\n               \'axis_x\': float(axis[0]), \'axis_y\': float(axis[1]),\n               \'frames\': len(t), \'gaps\': int(np.sum(~valid[1:])),\n               \'duration\': float(t[-1]), \'peak_speed\': float(np.max(speed))}\n    details = {\'t\': t, \'xy\': xy, \'s\': speed, \'acceleration\': acceleration,\n               \'lateral\': lateral, \'lateral_velocity\': velocity, \'segment\': segment,\n               \'reversals\': reversals}\n    return summary, details\n\n\ndef main():\n    source = pd.read_parquet(PROCESSED/\'combine_tracking_clean.parquet\')\n    players = pd.read_parquet(PROCESSED/\'players.parquet\')\n    ids = players.loc[players.nfl_position.isin([\'C\',\'G\',\'T\']), \'nfl_id\']\n    source = source.loc[source.nfl_id.isin(ids) & source.drill_name.isin(MIRROR_NAMES)].copy()\n    source.drill_name = \'PASS_PRO_MIRROR_DRILL\'\n    summaries, events = [], []\n    for key, group in source.groupby(KEYS, observed=True):\n        identity = dict(zip(KEYS, key))\n        summary, detail = measure_attempt(group)\n        summaries.append(identity | summary)\n        events.extend(identity | r for r in detail[\'reversals\'])\n    attempts = pd.DataFrame(summaries)\n    event_frame = pd.DataFrame(events)\n    destination = PROCESSED/\'ol\'\n    destination.mkdir(exist_ok=True)\n    attempts.to_parquet(destination/\'mirror_attempts.parquet\', index=False)\n    event_frame.to_parquet(destination/\'reversals.parquet\', index=False)\n    # Median attempts prevents players with more frames from dominating.\n    features = attempts.groupby([\'nfl_id\',\'draft_year\'], observed=True).agg(\n        lre_delay=(\'recovery_delay\',\'median\'), lre_completion=(\'completion_fraction\',\'mean\'),\n        mirror_peak_speed=(\'peak_speed\',\'median\'), mirror_attempts=(\'frames\',\'size\'),\n        detected_reversals=(\'reversals\',\'sum\'), completed_reversals=(\'completed\',\'sum\'),\n        axis_variance_fraction=(\'axis_variance_fraction\',\'median\')).reset_index()\n    features.to_parquet(destination/\'movement_features.parquet\', index=False)\n    repeated = int((features.mirror_attempts > 1).sum())\n    agreement = []\n    for _, group in event_frame.loc[event_frame.completed].groupby([\'nfl_id\',\'draft_year\'], observed=True):\n        values = group.delay.to_numpy(dtype=float)\n        if len(values) >= 4:\n            agreement.append([np.median(values[::2]),np.median(values[1::2])])\n    correlation = float(spearmanr(np.asarray(agreement)[:,0],np.asarray(agreement)[:,1]).statistic) if len(agreement)>3 else None\n    report = {\'mirror_players\':len(features),\'mirror_attempts\':len(attempts),\'mirror_frames\':int(attempts.frames.sum()),\n              \'detected_reversals\':len(event_frame),\'completed_reversals\':int(event_frame.completed.sum()),\n              \'players_with_delay\':int(features.lre_delay.notna().sum()),\'players_with_repeated_attempts\':repeated,\n              \'within_attempt_odd_even_players\':len(agreement),\'within_attempt_odd_even_spearman\':correlation,\n              \'repeatability_status\':\'Independent repeated-attempt reliability unavailable\' if repeated==0 else \'Repeated attempts available; reliability not yet established\',\n              \'candidate_definition\':\'Median observed recovery time to 20% of robust dominant-axis excursion; incomplete events retained separately.\'}\n    out = ROOT/\'outputs/research/ol\';out.mkdir(parents=True,exist_ok=True)\n    (out/\'movement_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'src/ol_outcomes.py': '"""Explicit pressure opportunity and censoring rules, kept separate from receivers."""\nimport json\nimport numpy as np\nimport pandas as pd\nfrom src.load_data import PROCESSED, ROOT\n\n\ndef flag(series):\n    return series.astype(\'string\').str.upper().map({\'TRUE\':1.,\'FALSE\':0.,\'1\':1.,\'0\':0.,\'1.0\':1.,\'0.0\':0.})\n\n\ndef construct_outcomes(plays,games,players):\n    ol=players.loc[players.nfl_position.isin([\'C\',\'G\',\'T\']),[\'nfl_id\',\'draft_year\',\'nfl_position\']]\n    merged=plays.merge(games[[\'game_id\',\'season\',\'season_type\']],on=\'game_id\',validate=\'many_to_one\').merge(ol,on=\'nfl_id\',validate=\'many_to_one\')\n    rookie=merged.loc[merged.season_type.eq(\'REG\') & merged.season.eq(merged.draft_year) & merged.play_nullified_by_penalty.eq(\'N\')].copy()\n    if rookie.duplicated([\'game_id\',\'play_id\',\'nfl_id\']).any():\n        raise ValueError(\'Duplicate OL player-play rows require resolution.\')\n    rookie[\'pressure\']=flag(rookie.pressure_allowed)\n    eligible=(rookie.lined_up_position.isin([\'C\',\'G\',\'T\']) & rookie.pressure.notna()\n              & rookie.dropback_duration.gt(0) & rookie.pass_rushers_encountered.ge(1)\n              & rookie.pass_result.notna())\n    obs=rookie.loc[eligible].copy()\n    event_time=pd.to_numeric(obs.time_to_pressure_allowed,errors=\'coerce\').astype(float)\n    obs[\'timing_valid\']=(obs.pressure.eq(0) | (event_time.ge(0) & event_time.le(obs.dropback_duration))).fillna(False)\n    obs[\'exit_time\']=np.where(obs.pressure.eq(1),event_time,obs.dropback_duration)\n    obs[\'event_by_3\']=obs.timing_valid & obs.pressure.eq(1) & event_time.le(3)\n    obs[\'observed_clean_at_3\']=obs.timing_valid & ~obs.event_by_3 & obs.exit_time.ge(3)\n    obs[\'censored_before_3\']=obs.timing_valid & ~obs.event_by_3 & obs.exit_time.lt(3)\n    def summarize(g):\n        long=g.loc[g.dropback_duration.ge(3)]\n        return pd.Series({\'opportunities\':len(g),\'pressures\':int(g.pressure.sum()),\'pressure_rate\':g.pressure.mean(),\n                          \'long_opportunities\':len(long),\'long_pressures\':int(long.pressure.sum()),\n                          \'long_pressure_rate\':long.pressure.mean(),\'timing_observations\':int(g.timing_valid.sum()),\n                          \'events_by_3\':int(g.event_by_3.sum()),\'observed_clean_at_3\':int(g.observed_clean_at_3.sum()),\n                          \'censored_before_3\':int(g.censored_before_3.sum()),\'mean_dropback_duration\':g.dropback_duration.mean(),\n                          \'mean_rushers\':g.pass_rushers_encountered.mean(),\n                          \'mean_pressure_time\':g.loc[g.pressure.eq(1)&g.timing_valid,\'exit_time\'].mean()})\n    summaries=obs.groupby([\'nfl_id\',\'draft_year\',\'nfl_position\'],observed=True).apply(summarize,include_groups=False).reset_index()\n    report={\'ol_prospects\':len(ol),\'rookie_regular_season_player_plays\':len(rookie),\n            \'proxy_protection_opportunities\':len(obs),\'players_with_opportunities\':int(obs.nfl_id.nunique()),\n            \'pressure_events\':int(obs.pressure.sum()),\'timing_inconsistent_or_missing\':int((~obs.timing_valid).sum()),\n            \'dropbacks_at_least_3_seconds\':int(obs.dropback_duration.ge(3).sum()),\n            \'censored_before_3\':int(obs.censored_before_3.sum()),\'observed_clean_at_3\':int(obs.observed_clean_at_3.sum()),\n            \'pressure_events_by_3\':int(obs.event_by_3.sum()),\n            \'survival_status\':\'Descriptive risk sets only. Release/end-of-dropback may be informative censoring; a validated survival prediction model is deferred.\'}\n    return obs,summaries,report\n\n\ndef main():\n    obs,summaries,report=construct_outcomes(*(pd.read_parquet(PROCESSED/f\'{name}.parquet\') for name in [\'player_play\',\'games\',\'players\']))\n    dest=PROCESSED/\'ol\';dest.mkdir(exist_ok=True)\n    obs.to_parquet(dest/\'opportunities.parquet\',index=False)\n    summaries.to_parquet(dest/\'outcomes.parquet\',index=False)\n    out=ROOT/\'outputs/research/ol\';out.mkdir(parents=True,exist_ok=True)\n    (out/\'outcome_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'src/ol_models.py': '"""Player-level temporal holdouts for the exploratory OL pressure-rate hypothesis."""\nimport json\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import expit,logit\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom src.load_data import PROCESSED, ROOT\nfrom src.train_models import BASELINE\n\n\ndef fit_comparison(data,year,seed=2027):\n    train=data.loc[data.draft_year.lt(year)].copy();test=data.loc[data.draft_year.eq(year)].copy()\n    if len(train)<10 or len(test)<5:return None,None\n    base=BASELINE+[\'role_C\',\'role_T\']\n    columns={\'traditional\':base,\'tracking\':base+[\'lre_delay\',\'lre_completion\']}\n    target=logit((train.pressures+.5)/(train.opportunities+1))\n    rows=test[[\'nfl_id\',\'draft_year\',\'nfl_position\',\'opportunities\',\'pressures\',\'pressure_rate\',\'long_opportunities\',\'long_pressure_rate\',\'lre_delay\',\'lre_completion\']].copy()\n    models={};effects={}\n    for name,features in columns.items():\n        pipeline=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True),StandardScaler(),Ridge(alpha=10.))\n        pipeline.fit(train[features],target)\n        prediction=expit(pipeline.predict(test[features]))\n        rows[\'prediction_\'+name]=prediction\n        n=test.opportunities.to_numpy(dtype=float);events=test.pressures.to_numpy(dtype=float)\n        models[name]={\'mae\':float(np.mean(np.abs(prediction-test.pressure_rate))),\n                      \'opportunity_brier\':float(np.sum(n*prediction**2-2*prediction*events+events)/n.sum()),\'features\':features}\n        effects[name]=dict(zip(features,pipeline[-1].coef_[:len(features)].astype(float).tolist()))\n    null=float(train.pressures.sum()/train.opportunities.sum())\n    rows[\'prediction_train_rate\']=null\n    improvement=(np.abs(rows.prediction_traditional-rows.pressure_rate)-np.abs(rows.prediction_tracking-rows.pressure_rate)).to_numpy(dtype=float)\n    rng=np.random.default_rng(seed);boot=rng.choice(improvement,size=(2000,len(improvement))).mean(axis=1)\n    report={\'experiment\':\'offensive_line_pressure_rate\',\'held_out_draft_year\':int(year),\'train_players\':len(train),\'test_players\':len(test),\n            \'train_opportunities\':int(train.opportunities.sum()),\'test_opportunities\':int(test.opportunities.sum()),\n            \'models\':models,\'train_rate_benchmark_mae\':float(np.mean(np.abs(null-test.pressure_rate))),\n            \'mae_improvement\':float(improvement.mean()),\'mae_improvement_bootstrap_95pct\':np.quantile(boot,[.025,.975]).tolist(),\n            \'standardized_logit_coefficients\':effects,\'status\':\'Exploratory temporal holdout; LRE not validated\',\n            \'limitations\':[\'Small player sample. Fixed regularization. Conditional deployment and opportunity selection.\',\n                           \'Bootstrap samples held-out players and does not include model refitting or game clustering.\',\n                           \'Completed-event median can be selective. Lateral axis is a geometric proxy.\',\n                           \'Predicts any-pressure rate, not causal three-second protection or a survival hazard.\']}\n    return report,rows\n\n\ndef main():\n    outcomes=pd.read_parquet(PROCESSED/\'ol/outcomes.parquet\')\n    features=pd.read_parquet(PROCESSED/\'ol/movement_features.parquet\')\n    combine=pd.read_parquet(PROCESSED/\'combine_results.parquet\')\n    data=outcomes.merge(features,on=[\'nfl_id\',\'draft_year\'],validate=\'one_to_one\').merge(combine,on=[\'nfl_id\',\'draft_year\'],validate=\'one_to_one\')\n    data=data.loc[data.opportunities.ge(50)&data.lre_delay.notna()&data.detected_reversals.ge(2)].copy()\n    data[\'role_C\']=data.nfl_position.eq(\'C\').astype(int);data[\'role_T\']=data.nfl_position.eq(\'T\').astype(int)\n    out=ROOT/\'outputs/research/ol\';out.mkdir(parents=True,exist_ok=True)\n    data.to_parquet(PROCESSED/\'ol/model_cohort.parquet\',index=False)\n    reports=[]\n    for year in sorted(data.draft_year.unique())[1:]:\n        report,rows=fit_comparison(data,year)\n        if report:\n            reports.append(report);rows.to_csv(out/f\'predictions_{int(year)}.csv\',index=False)\n    if not reports:\n        (out/\'model_comparison.json\').write_text(json.dumps({\'status\':\'Analysis incomplete\',\'reason\':\'Insufficient eligible temporal cohort sizes\',\'eligible_players\':len(data)},indent=2))\n        return\n    report=reports[-1];report[\'temporal_splits\']=reports[:-1]\n    report[\'eligible_players\']=len(data)\n    (out/\'model_comparison.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    print(json.dumps({k:v for k,v in report.items() if k not in [\'standardized_logit_coefficients\',\'temporal_splits\',\'models\']},indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'src/ol_sensitivity.py': '"""Probe the provisional recovery-distance choice without selecting a winner."""\nimport json\nimport pandas as pd\nfrom src.load_data import ROOT,PROCESSED\nfrom src.clean_tracking import KEYS\nfrom src.ol_movement import measure_attempt,MIRROR_NAMES\nfrom src.ol_models import fit_comparison\n\n\ndef main():\n    frame=pd.read_parquet(PROCESSED/\'combine_tracking_clean.parquet\')\n    cohort=pd.read_parquet(PROCESSED/\'ol/model_cohort.parquet\')\n    frame=frame.loc[frame.nfl_id.isin(cohort.nfl_id)&frame.drill_name.isin(MIRROR_NAMES)].copy()\n    frame.drill_name=\'PASS_PRO_MIRROR_DRILL\'\n    reports=[]\n    for fraction in [.15,.2,.25]:\n        values=[]\n        for key,group in frame.groupby(KEYS,observed=True):\n            measured,_=measure_attempt(group,fraction)\n            values.append({\'nfl_id\':key[1],\'draft_year\':key[0],\'lre_delay\':measured[\'recovery_delay\'],\'lre_completion\':measured[\'completion_fraction\']})\n        replacement=pd.DataFrame(values).groupby([\'nfl_id\',\'draft_year\'],as_index=False).median()\n        data=cohort.drop(columns=[\'lre_delay\',\'lre_completion\']).merge(replacement,on=[\'nfl_id\',\'draft_year\'],validate=\'one_to_one\')\n        data=data.loc[data.lre_delay.notna()]\n        result,_=fit_comparison(data,data.draft_year.max())\n        reports.append({\'excursion_fraction\':fraction,\'eligible_players\':len(data),\n                        \'median_recovery_delay\':float(data.lre_delay.median()),\n                        \'mae_improvement\':result[\'mae_improvement\'],\n                        \'interval\':result[\'mae_improvement_bootstrap_95pct\']})\n    (ROOT/\'outputs/research/ol/sensitivity.json\').write_text(json.dumps(reports,indent=2),encoding=\'utf-8\')\n    print(json.dumps(reports,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'scripts/__init__.py': '"""Reproducible local build and preview tools."""\n', 'scripts/build_presentation.py': '"""Export local-only research evidence for the modular presentation. Never deploy this export."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\nfrom src.load_data import ROOT,PROCESSED\nfrom src.clean_tracking import KEYS\nfrom src.ol_movement import measure_attempt,MIRROR_NAMES\n\n\ndef safe(value):\n    if isinstance(value,dict):return {str(k):safe(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple,np.ndarray)):return [safe(v) for v in value]\n    if isinstance(value,(np.integer,)):return int(value)\n    if isinstance(value,(float,np.floating)):return round(float(value),6) if np.isfinite(value) else None\n    if value is pd.NA or value is pd.NaT:return None\n    if isinstance(value,(np.bool_,)):return bool(value)\n    return value\n\n\ndef main():\n    players=pd.read_parquet(PROCESSED/\'players.parquet\')\n    cohort=pd.read_parquet(PROCESSED/\'ol/model_cohort.parquet\').merge(players[[\'nfl_id\',\'display_name\']],on=\'nfl_id\',validate=\'one_to_one\')\n    position=cohort.nfl_position.value_counts().index[0]\n    candidates=cohort.loc[cohort.nfl_position.eq(position)].sort_values(\'nfl_id\')\n    cases=[];selected=set()\n    for percentile in [.25,.5,.75]:\n        target=candidates.lre_delay.quantile(percentile)\n        choice=candidates.loc[~candidates.nfl_id.isin(selected)].assign(distance=lambda d:(d.lre_delay-target).abs()).sort_values([\'distance\',\'nfl_id\']).iloc[0]\n        selected.add(choice.nfl_id)\n        row=choice[[\'nfl_id\',\'display_name\',\'nfl_position\',\'draft_year\',\'lre_delay\',\'lre_completion\',\'opportunities\',\'pressure_rate\',\'long_opportunities\',\'long_pressure_rate\']].to_dict()\n        row[\'selection_percentile\']=percentile;cases.append(row)\n    out=ROOT/\'outputs/research/ol\'\n    (out/\'case_selection.json\').write_text(json.dumps(safe({\'procedure\':\'Most common eligible OL position; nearest 25th/50th/75th recovery-delay quantile; ties by NFL ID; outcomes not used.\',\'position\':position,\'cases\':cases}),indent=2))\n    source=pd.read_parquet(PROCESSED/\'combine_tracking_clean.parquet\')\n    ol=players.loc[players.nfl_position.isin([\'C\',\'G\',\'T\'])]\n    drills=MIRROR_NAMES|{\'PASS_RUSH_DROP\',\'FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE\'}\n    source=source.loc[source.nfl_id.isin(ol.nfl_id)&source.drill_name.isin(drills)].copy()\n    source.loc[source.drill_name.isin(MIRROR_NAMES),\'drill_name\']=\'PASS_PRO_MIRROR_DRILL\'\n    metadata=players.set_index(\'nfl_id\').to_dict(orient=\'index\')\n    trajectories=[];worked=[]\n    for key,group in source.groupby(KEYS,observed=True):\n        group=group.sort_values(\'time\')\n        summary,detail=measure_attempt(group)\n        player=metadata[key[1]]\n        record={\'id\':f\'{int(key[1])}-{key[2]}-{int(key[4])}\',\'nfl_id\':int(key[1]),\'year\':int(key[0]),\n                \'name\':player[\'display_name\'],\'position\':player[\'nfl_position\'],\'drill\':key[3],\'attempt\':int(key[4]),\n                \'event_id\':key[2],\'summary\':summary,\n                \'frames\':np.column_stack([detail[\'t\'],detail[\'xy\'],detail[\'s\'],detail[\'acceleration\'],detail[\'lateral\'],detail[\'lateral_velocity\'],detail[\'segment\']]),\n                \'reversals\':detail[\'reversals\'] if key[3]==\'PASS_PRO_MIRROR_DRILL\' else []}\n        trajectories.append(record)\n        if key[1] in selected and key[3]==\'PASS_PRO_MIRROR_DRILL\':\n            variants=[]\n            for fraction in [.15,.2,.25]:\n                measured,variant=measure_attempt(group,fraction)\n                variants.append({\'fraction\':fraction,\'summary\':measured,\'reversals\':variant[\'reversals\']})\n            worked.append({\'nfl_id\':int(key[1]),\'variants\':variants})\n    # Stable order puts documented case-study athletes first, then player ID.\n    trajectories.sort(key=lambda t:(t[\'nfl_id\'] not in selected,t[\'nfl_id\'],t[\'drill\'],t[\'attempt\']))\n    manifest=json.loads((PROCESSED/\'ingestion_manifest.json\').read_text())\n    samples={}\n    for name in manifest:\n        raw=ROOT/\'data/raw\'/name\n        sample=pd.read_csv(raw,nrows=5) if raw.exists() else next(pq.ParquetFile(PROCESSED/(Path(name).stem+\'.parquet\')).iter_batches(batch_size=5)).to_pandas()\n        cols=list(sample.columns[:8])\n        if \'tracking\' in name:\n            cols=[c for c in [\'nfl_id\',\'time\',\'drill_name\',\'play_id\',\'x\',\'y\',\'s\',\'a\'] if c in sample.columns]\n        samples[name]={\'columns\':cols,\'rows\':json.loads(sample[cols].to_json(orient=\'values\')),\'column_count\':len(sample.columns)}\n    ol_report=json.loads((out/\'model_comparison.json\').read_text())\n    prediction_file=out/f\'predictions_{ol_report["held_out_draft_year"]}.csv\'\n    predictions=pd.read_csv(prediction_file).merge(players[[\'nfl_id\',\'display_name\']],on=\'nfl_id\').to_dict(orient=\'records\')\n    receiver_predictions=pd.read_csv(ROOT/\'outputs/results/held_out_predictions.csv\').merge(players[[\'nfl_id\',\'display_name\']],on=\'nfl_id\').to_dict(orient=\'records\')\n    obs=pd.read_parquet(PROCESSED/\'ol/opportunities.parquet\')\n    duration_bins=[]\n    for left,right,label in [(0,2,\'Under 2 s\'),(2,3,\'2–3 s\'),(3,4,\'3–4 s\'),(4,100,\'4 s or longer\')]:\n        g=obs.loc[obs.dropback_duration.ge(left)&obs.dropback_duration.lt(right)]\n        duration_bins.append({\'label\':label,\'opportunities\':len(g),\'pressure_rate\':float(g.pressure.mean())})\n    files=[out/\'model_comparison.json\',ROOT/\'outputs/results/model_comparison.json\',out/\'movement_report.json\',out/\'outcome_report.json\']\n    payload={\'schema_version\':1,\'local_only\':True,\'raw_data_warning\':\'Contains restricted local trajectory examples. Do not commit or deploy.\',\n             \'manifest\':manifest,\'counts\':{\'prospects\':len(players),\'combine_attempts\':len(pd.read_parquet(PROCESSED/\'combine_attempt_metrics.parquet\')),\n             \'combine_clean_frames\':len(pd.read_parquet(PROCESSED/\'combine_tracking_clean.parquet\'))},\n             \'ol\':ol_report,\'receiver\':json.loads((ROOT/\'outputs/results/model_comparison.json\').read_text()),\n             \'movement\':json.loads((out/\'movement_report.json\').read_text()),\'outcomes\':json.loads((out/\'outcome_report.json\').read_text()),\n             \'sensitivity\':json.loads((out/\'sensitivity.json\').read_text()),\'cases\':cases,\'worked_examples\':worked,\n             \'trajectories\':trajectories,\'samples\':samples,\'ol_predictions\':predictions,\'receiver_predictions\':receiver_predictions,\n             \'duration_bins\':duration_bins,\n             \'provenance\':{str(f.relative_to(ROOT)):hashlib.sha256(f.read_bytes()).hexdigest() for f in files}}\n    destination=ROOT/\'outputs/presentation\';destination.mkdir(parents=True,exist_ok=True)\n    (destination/\'research.json\').write_text(json.dumps(safe(payload),separators=(\',\',\':\'),allow_nan=False),encoding=\'utf-8\')\n    # Compact browser entry point: frames are fetched only for selected attempts.\n    outcomes=pd.read_parquet(PROCESSED/\'ol/outcomes.parquet\').set_index(\'nfl_id\').to_dict(orient=\'index\')\n    attempts_dir=destination/\'attempts\';attempts_dir.mkdir(exist_ok=True)\n    catalog=[]\n    for trajectory in trajectories:\n        filename=hashlib.sha256(trajectory[\'id\'].encode()).hexdigest()[:20]+\'.json\'\n        trajectory[\'nfl\']=outcomes.get(trajectory[\'nfl_id\'])\n        (attempts_dir/filename).write_text(json.dumps(safe(trajectory),separators=(\',\',\':\'),allow_nan=False),encoding=\'utf-8\')\n        item={k:v for k,v in trajectory.items() if k not in [\'frames\',\'reversals\']}\n        item[\'file\']=\'attempts/\'+filename;item[\'frame_count\']=len(trajectory[\'frames\'])\n        catalog.append(item)\n    briefing={k:v for k,v in payload.items() if k not in [\'trajectories\',\'samples\',\'worked_examples\']}\n    briefing[\'schema_version\']=2\n    briefing[\'featured_attempts\']=[next(t for t in catalog if t[\'nfl_id\']==c[\'nfl_id\'] and t[\'drill\']==\'PASS_PRO_MIRROR_DRILL\') for c in cases]\n    briefing[\'model_opportunities\']=ol_report[\'train_opportunities\']+ol_report[\'test_opportunities\']\n    for filename,value in [(\'briefing.json\',briefing),(\'catalog.json\',catalog),(\'appendix.json\',{\'samples\':samples,\'worked_examples\':worked})]:\n        (destination/filename).write_text(json.dumps(safe(value),separators=(\',\',\':\'),allow_nan=False),encoding=\'utf-8\')\n    print(f\'Exported {len(trajectories)} local attempts; compact briefing plus on-demand trajectory files.\')\n\n\nif __name__==\'__main__\':main()\n', 'scripts/research_checks.py': '"""Verify outputs, join invariants, cohort ordering, and preserved receiver scores."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom src.load_data import ROOT,PROCESSED\n\n\ndef main():\n    messages=[]\n    def check(ok,label):\n        if not ok:raise AssertionError(label)\n        messages.append(label)\n    current=json.loads((ROOT/\'outputs/results/model_comparison.json\').read_text())\n    saved=json.loads((ROOT/\'outputs/checkpoints/pre-overhaul/results/model_comparison.json\').read_text()) if (ROOT/\'outputs/checkpoints/pre-overhaul/results/model_comparison.json\').exists() else {\'mae_improvement\':-0.2947473905667486}\n    check(np.isclose(current[\'mae_improvement\'],saved[\'mae_improvement\']),\'Original receiver MAE comparison reproduced\')\n    events=pd.read_parquet(PROCESSED/\'ol/reversals.parquet\')\n    check(events.loc[events.completed,\'delay\'].gt(0).all(),\'Completed recoveries have positive observed delay\')\n    check(events.loc[~events.completed,\'delay\'].isna().all(),\'Incomplete recoveries remain censored\')\n    obs=pd.read_parquet(PROCESSED/\'ol/opportunities.parquet\')\n    check(not obs.duplicated([\'game_id\',\'play_id\',\'nfl_id\']).any(),\'OL opportunities have unique player-play keys\')\n    check(obs.season.eq(obs.draft_year).all() & obs.season_type.eq(\'REG\').all(),\'OL outcomes are rookie regular season only\')\n    check(obs.lined_up_position.isin([\'C\',\'G\',\'T\']).all() & obs.pass_rushers_encountered.ge(1).all(),\'Proxy opportunity role and rusher rules hold\')\n    count=obs.event_by_3.astype(int)+obs.observed_clean_at_3.astype(int)+obs.censored_before_3.astype(int)\n    check(count.loc[obs.timing_valid].eq(1).all(),\'Every valid timing row belongs to exactly one three-second risk set\')\n    check(count.loc[~obs.timing_valid].eq(0).all(),\'Invalid timing rows excluded from all time-based categories\')\n    report=json.loads((ROOT/\'outputs/research/ol/model_comparison.json\').read_text())\n    cohort=pd.read_parquet(PROCESSED/\'ol/model_cohort.parquet\')\n    check(cohort.loc[cohort.draft_year.lt(report[\'held_out_draft_year\'])].nfl_id.nunique()==report[\'train_players\'],\'Training players precede held-out draft cohort\')\n    check(cohort.loc[cohort.draft_year.eq(report[\'held_out_draft_year\'])].nfl_id.nunique()==report[\'test_players\'],\'Held-out players match report\')\n    output=ROOT/\'outputs/verification\';output.mkdir(parents=True,exist_ok=True)\n    (output/\'research_checks.json\').write_text(json.dumps({\'passed\':len(messages),\'checks\':messages},indent=2),encoding=\'utf-8\')\n    print(\'\\n\'.join(\'PASS \'+m for m in messages))\n\n\nif __name__==\'__main__\':main()\n', 'visualizations/__init__.py': '"""Charts and animations for the analysis."""\n', 'visualizations/research_figures.py': '"""High-resolution research figures from the verified local export; no remote assets."""\nimport json\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom src.load_data import ROOT\nCYAN=\'#067c87\';GRAY=\'#747b83\';AMBER=\'#b77820\'\n\ndef save(fig,name):\n    dest=ROOT/\'outputs/figures\';dest.mkdir(parents=True,exist_ok=True)\n    fig.savefig(dest/f\'{name}.png\',dpi=240,bbox_inches=\'tight\')\n    fig.savefig(dest/f\'{name}.svg\',bbox_inches=\'tight\');plt.close(fig)\n\ndef main():\n    d=json.loads((ROOT/\'outputs/presentation/research.json\').read_text())\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'figure.facecolor\':\'white\'})\n    for receiver in [False,True]:\n        rows=d[\'receiver_predictions\'] if receiver else d[\'ol_predictions\'];actual=\'mean_separation\' if receiver else \'pressure_rate\';enh=\'prediction_traditional_plus_tracking\' if receiver else \'prediction_tracking\'\n        fig,ax=plt.subplots(figsize=(6.7,5.1));x=np.array([r[actual] for r in rows]);b=np.array([r[\'prediction_traditional\'] for r in rows]);e=np.array([r[enh] for r in rows]);ax.vlines(x,np.minimum(b,e),np.maximum(b,e),color=\'#cccccc\',lw=1);ax.scatter(x,b,color=GRAY,label=\'Traditional\',s=24);ax.scatter(x,e,color=CYAN,label=\'Tracking enhanced\',s=24);lim=max(x.max(),b.max(),e.max())*1.07;ax.plot([0,lim],[0,lim],color=\'#aaaaaa\',ls=\'--\',label=\'Identity\');ax.set(xlim=(0,lim),ylim=(0,lim),xlabel=\'Observed mean separation (source units)\' if receiver else \'Observed player pressure rate\',ylabel=\'Predicted outcome\',title=f\'{"Receiver" if receiver else "Offensive-line"} experiment / 2025 temporal holdout (n={len(rows)})\');ax.legend(frameon=False);fig.text(.12,-.01,\'Separate hypotheses. Player-level errors; observational outcomes.\',fontsize=9,color=GRAY);save(fig,\'receiver_predictions\' if receiver else \'ol_predictions\')\n    fig,axes=plt.subplots(1,2,figsize=(10.4,4.2));o=d[\'outcomes\'];keys=[\'pressure_events_by_3\',\'observed_clean_at_3\',\'censored_before_3\',\'timing_inconsistent_or_missing\'];axes[0].barh([\'Pressure by 3 s\',\'Clean through 3 s\',\'Censored before 3 s\',\'Timing excluded\'],[o[k] for k in keys],color=[AMBER,CYAN,GRAY,\'#bdbdbd\']);axes[0].invert_yaxis();axes[0].set(xlabel=\'Proxy opportunities\',title=\'Disjoint timing classifications\');axes[1].bar(range(4),[x[\'pressure_rate\']*100 for x in d[\'duration_bins\']],color=CYAN);axes[1].set_xticks(range(4),[x[\'label\'] for x in d[\'duration_bins\']],rotation=20,ha=\'right\');axes[1].set(ylabel=\'Any-pressure rate (%)\',title=\'Conditional on observed dropback duration\');fig.suptitle(\'Short pressure-free dropbacks do not establish three-second resilience\');fig.tight_layout();fig.text(.1,-.08,\'Descriptive, not causal. Release censoring, role and deployment remain unresolved.\',fontsize=9,color=GRAY);save(fig,\'nfl_observation_boundary\')\n    fig,ax=plt.subplots(figsize=(7,4.2));s=d[\'sensitivity\'];y=np.arange(len(s));value=np.array([r[\'mae_improvement\'] for r in s]);lo=np.array([r[\'interval\'][0] for r in s]);hi=np.array([r[\'interval\'][1] for r in s]);ax.errorbar(value*100,y,xerr=np.vstack([value-lo,hi-value])*100,fmt=\'o\',color=CYAN,capsize=4);ax.axvline(0,color=GRAY,ls=\'--\');ax.set_yticks(y,[f"{r[\'excursion_fraction\']:.0%} excursion target" for r in s]);ax.set(xlabel=\'Traditional MAE − tracking MAE (percentage points)\',title=\'OL metric sensitivity / paired test-player bootstrap 95% intervals\');fig.text(.12,-.03,\'2025 test n=17. No target was selected using these held-out results.\',fontsize=9,color=GRAY);save(fig,\'lre_sensitivity\')\n    t=next(t for t in d[\'trajectories\'] if t[\'nfl_id\']==d[\'cases\'][0][\'nfl_id\'] and t[\'reversals\']);f=np.array([[np.nan if v is None else v for v in r] for r in t[\'frames\']],dtype=float);fig,axes=plt.subplots(2,1,figsize=(8,5.5),sharex=True);axes[0].plot(f[:,0],f[:,5],color=CYAN,label=\'Projected displacement\');axes[1].plot(f[:,0],f[:,6],color=CYAN,label=\'Finite-difference velocity\');\n    for i,r in enumerate(t[\'reversals\']):\n        for ax in axes:ax.axvline(r[\'time\'],color=AMBER,ls=\':\',lw=1)\n        axes[0].scatter(r[\'time\'],f[r[\'index\'],5],marker=\'D\',color=AMBER,zorder=3)\n        if r[\'completed\']:axes[0].plot([r[\'time\'],f[r[\'recovery_index\'],0]],[f[r[\'index\'],5]]*2,color=AMBER,lw=3);axes[0].annotate(f"R{i+1}: {r[\'delay\']:.2f}s",(r[\'time\'],f[r[\'index\'],5]),xytext=(4,9),textcoords=\'offset points\',fontsize=8)\n    axes[0].set(ylabel=\'Projected position (source units)\',title=f"Candidate recovery / {t[\'name\']} / mirror attempt");axes[0].margins(y=.25);axes[1].set(xlabel=\'Elapsed time (s)\',ylabel=\'Axis velocity (source units/s)\');axes[1].axhline(0,color=GRAY,lw=.8);fig.tight_layout();fig.text(.1,-.03,\'Dominant-axis proxy; no body orientation. Target = 20% of p90–p10 excursion.\',fontsize=9,color=GRAY);save(fig,\'lre_worked_example\')\n    print(\'Wrote five scientific figures in PNG and SVG.\')\nif __name__==\'__main__\':main()\n'}
SOURCE_HASHES = {'src/__init__.py': '9fc4dbf1cfb5daf54229d8f2787be4376789e23cd8c977a30295efbaadd8376b', 'src/load_data.py': '10444ecdee0c9a46c782237df5ac45baa13c047df04e59abcdf25d815a72724d', 'src/clean_tracking.py': '448799f12c71479c5bcca51eb942c54986cae5f1716962b6ff004bb2ecc62ba0', 'src/combine_metrics.py': '41094e70d181e62d60d425587b9cd200c304c7dbdbe05ad06b604036bc8dc418', 'src/nfl_outcomes.py': '7d016364a0fc172cc78ba43f98472888064029a3467f0281b1a19251b961abc2', 'src/train_models.py': 'ad513005f360636a86ee3fba2aa0997be6a055a62d10870b456ad15542b54e69', 'src/validate_results.py': 'eb63855d9cb2d53025dedb6233946cf9e40dc38e645ea09dc89905fd25aefc15', 'src/ol_movement.py': '9bbf2c1d89c8d9080bc6b66c0f6d182ba78c8ae0ebcb85da03ac057dcb482fa4', 'src/ol_outcomes.py': '4e135ef58ae7d2c27a09ae24a8fd1206748e972dcfb93a1869b92cdca558eabf', 'src/ol_models.py': '31b1f69f25b699f74bca3a5051fbfc12a084e3c301082d39f5d6df8d305df6b4', 'src/ol_sensitivity.py': '434223ab8bbe5847062e4952c522f95d0f0beed8017c08b55ede70cbf58ca163', 'scripts/__init__.py': '0f12aede2f929863b847599dbb1a25d9fe0cf1c506f33a5666a9d33d74144eb8', 'scripts/build_presentation.py': '32937595b1f3990baf3b2db119050ef6d652d428ef2fedbfaab9e46e53d5c2d4', 'scripts/research_checks.py': '7c305a08932644ee63d18e58cde0d3dcb1e794cae486dedec5aab6c5766b7b33', 'visualizations/__init__.py': '3ac015589d13439f0a6dd8fdaacfab6a5d229a0b8696fc5368bb4a2e139501ab', 'visualizations/research_figures.py': '75bf696c1c1a23a046b1f37dd66c51585904393538b87d86a5456437d130bc02'}
for name, text in SOURCES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == SOURCE_HASHES[name]
    target = WORK / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8")
sys.path.insert(0, str(WORK))


In [ ]:
# Locate the official dataset root; fail on ambiguity rather than mix sources.
if IN_KAGGLE:
    candidates = [p.parent for p in Path("/kaggle/input").rglob("combine_tracking.csv") if (p.parent / "player_play.csv").exists()]
else:
    candidates = [p for p in [Path.cwd()/"data/raw", Path.cwd().parent/"data/raw"] if (p/"combine_tracking.csv").exists()]
assert len(candidates) == 1, f"Expected one official dataset root, found {candidates}"
SOURCE = candidates[0]
# Only five tables are needed; the 22.7M game-tracking rows are not required.
required = ["players", "games", "combine_results", "combine_tracking", "player_play"]
from src.load_data import convert_csv, PROCESSED
PROCESSED.mkdir(parents=True, exist_ok=True)
manifest = {}
for name in required:
    csv = SOURCE / f"{name}.csv"
    assert csv.exists(), csv
    rows = convert_csv(csv, PROCESSED / f"{name}.parquet")
    manifest[csv.name] = {"rows": rows, "source_bytes": csv.stat().st_size}
    print(name, rows)
(PROCESSED/"ingestion_manifest.json").write_text(json.dumps(manifest))


## Prespecified pipeline

The default OL recovery target is 20% of within-attempt p90–p10 projected excursion. Missing recoveries remain censored. Only earlier cohorts train the models. Alternate 15% and 25% targets assess sensitivity and do not select a winner. Short dropbacks do not establish protection at three seconds.


In [ ]:
def run(module):
    result = subprocess.run([sys.executable, "-m", module], cwd=WORK, text=True, capture_output=True)
    print(result.stdout[-7000:])
    if result.returncode:
        raise RuntimeError(result.stderr)
for module in ["src.clean_tracking", "src.combine_metrics", "src.nfl_outcomes", "src.validate_results", "src.train_models", "src.ol_movement", "src.ol_outcomes", "src.ol_models", "src.ol_sensitivity"]:
    print("RUN", module)
    run(module)


In [ ]:
receiver = json.loads((WORK/"outputs/results/model_comparison.json").read_text())
ol = json.loads((WORK/"outputs/research/ol/model_comparison.json").read_text())
for label, report in [("WR/TE separation", receiver), ("OL pressure rate", ol)]:
    print(label, json.dumps(report, indent=2)[:6500])
assert ol["held_out_draft_year"] == 2025
assert ol["train_players"] == 37 and ol["test_players"] == 17
assert abs(receiver["models"]["traditional"]["mae"] - 0.24906361836797772) < 1e-7
assert abs(ol["models"]["traditional"]["mae"] - 0.023368755233743682) < 1e-7
# Numeric comparisons can need minor tolerance changes across dependency versions.


In [ ]:
import pandas as pd
obs = pd.read_parquet(WORK/"data/processed/ol/opportunities.parquet")
classification = obs.event_by_3.astype(int)+obs.observed_clean_at_3.astype(int)+obs.censored_before_3.astype(int)
assert classification[obs.timing_valid].eq(1).all()
assert classification[~obs.timing_valid].eq(0).all()
assert obs.season.eq(obs.draft_year).all() and obs.season_type.eq("REG").all()
assert not obs.duplicated(["game_id","play_id","nfl_id"]).any()
print("Independent opportunity and timing checks passed.")


In [ ]:
# Local export supports scientific plotting; it contains restricted trajectories.
# Do not include research.json or processed/raw files in a public notebook output.
run("scripts.build_presentation")
run("visualizations.research_figures")
from IPython.display import display, Image
for name in ["ol_predictions", "lre_worked_example", "lre_sensitivity", "nfl_observation_boundary", "receiver_predictions"]:
    display(Image(filename=str(WORK/"outputs/figures"/f"{name}.png")))


## Interpretation

Candidate LRE fails to establish incremental predictive value. The 2025 player-MAE difference is slightly negative and its bootstrap interval crosses zero; the opportunity Brier difference is tiny in the opposite direction. Both OL models lose to the pooled-training-rate benchmark on player MAE. The receiver tracking model performs substantially worse than its traditional baseline.

Limitations include small player cohorts, one mirror attempt per athlete, a geometric lateral proxy, unverified field semantics, conditional deployment, unmodeled scheme/quarterback context, informative release censoring, fixed regularization, and uncertainty conditional on fitted models. These results do not justify ranking prospects.


In [ ]:
# Remove reconstructable research outputs from Kaggle's public output bundle.
# Keep rendered notebook figures and aggregate model reports only.
if IN_KAGGLE:
    import shutil
    for folder in [WORK/"data", WORK/"outputs/presentation"]:
        if folder.exists():
            assert folder.resolve().is_relative_to(WORK.resolve())
            shutil.rmtree(folder)
    for filename in (WORK/"outputs").rglob("predictions_*.csv"):
        filename.unlink()
    receiver_predictions = WORK/"outputs/results/held_out_predictions.csv"
    receiver_predictions.unlink(missing_ok=True)
    print("Removed local trajectory export, processed data, and per-player prediction CSVs.")
